# ARC-LeWM: decode-by-planning JEPA, meta-trained on the ARC public train split
Spec: docs/superpowers/specs/2026-06-09-arc-lewm-design.md
LeWM methods kept verbatim: end-to-end (no EMA / no stop-grad), loss = pred MSE + 0.09*SIGReg, AdaLN predictor, inference = latent-cost optimization (no decoder).
Flagged deviation: K=16 register tokens instead of single CLS (validated by Gate L3).

In [1]:
import json, math, os, random, copy, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange, repeat
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MAX_HW = 30; NUM_COLORS = 10; PAD_ID = 10; VOCAB = 11
D = 192          # LeWM embed_dim
K = 16           # register tokens (flagged deviation from single CLS)
ND_MAX = 4       # max demo pairs per episode
LAM = 0.09       # LeWM sigreg weight
CKPT = Path("data/arc_lewm.pt")
print("device:", DEVICE, "| torch:", torch.__version__)

device: cuda | torch: 2.8.0+cu128


In [2]:
ARC_DIR = Path("data/arc")

def load_arc_split(split):
    """Return dict task_id -> {'train': [(in,out),...], 'test': [(in,out),...]}."""
    out = {}
    for p in sorted((ARC_DIR / "data" / split).glob("*.json")):
        d = json.loads(p.read_text())
        out[p.stem] = {
            "train": [(np.array(e["input"], np.uint8), np.array(e["output"], np.uint8)) for e in d["train"]],
            "test":  [(np.array(e["input"], np.uint8), np.array(e["output"], np.uint8)) for e in d["test"]],
        }
    return out

TRAIN_TASKS = load_arc_split("training")
EVAL_TASKS  = load_arc_split("evaluation")
assert len(TRAIN_TASKS) == 400 and len(EVAL_TASKS) == 400
print("OK: 400 train / 400 eval tasks")

OK: 400 train / 400 eval tasks
